# Exploration of the Klinikatlas API

## Retrieve ICD Codes, OPS Codes and basic hospital information

In [ ]:
from deutschland import klinikatlas
from deutschland.klinikatlas.api import default_api
import requests

BASE_URL = "https://bundes-klinik-atlas.de"

configuration = klinikatlas.Configuration(
    host=BASE_URL
)

with klinikatlas.ApiClient(configuration) as api_client:
    api = default_api.DefaultApi(api_client)

    icd_codes = api.fileadmin_json_icd_codes_json_get()
    ops_codes = api.fileadmin_json_ops_codes_json_get()
    locations = api.fileadmin_json_locations_json_get()


In [31]:
icd_codes[:5]

[{'description': 'Cholera durch Vibrio cholerae O:1, Biovar cholerae',
  'icdcode': 'A00.0'},
 {'description': 'Cholera durch Vibrio cholerae O:1, Biovar eltor',
  'icdcode': 'A00.1'},
 {'description': 'Cholera, nicht näher bezeichnet', 'icdcode': 'A00.9'},
 {'description': 'Typhus abdominalis und Paratyphus', 'icdcode': 'A01'},
 {'description': 'Typhus abdominalis', 'icdcode': 'A01.0'}]

In [32]:
ops_codes[:5]

[{'description': 'Diagnostik zur Feststellung des irreversiblen '
                 'Hirnfunktionsausfalls',
  'opscode': '1-202'},
 {'description': 'Diagnostik zur Feststellung des irreversiblen '
                 'Hirnfunktionsausfalls: Bei einem potenziellen Organspender',
  'opscode': '1-202.0'},
 {'description': 'Diagnostik zur Feststellung des irreversiblen '
                 'Hirnfunktionsausfalls: Bei einem potenziellen Organspender: '
                 'Ohne Feststellung des irreversiblen Hirnfunktionsausfalls',
  'opscode': '1-202.00'},
 {'description': 'Diagnostik zur Feststellung des irreversiblen '
                 'Hirnfunktionsausfalls: Bei einem potenziellen Organspender: '
                 'Mit Feststellung des irreversiblen Hirnfunktionsausfalls',
  'opscode': '1-202.01'},
 {'description': 'Diagnostik zur Feststellung des irreversiblen '
                 'Hirnfunktionsausfalls: Bei sonstigen Patienten',
  'opscode': '1-202.1'}]

In [33]:
locations[:5]

[{'beds_number': 533,
  'city': 'Rostock',
  'latitude': '54.071629513465',
  'link': 'https://bundes-klinik-atlas.de/krankenhaussuche/krankenhaus/771003/',
  'longitude': '12.107577323914',
  'mail': 'info@kliniksued-rostock.de',
  'name': 'Klinikum Südstadt Rostock',
  'phone': '+49 (0)381/4401-0',
  'street': 'Südring 81',
  'zip': '18059'},
 {'beds_number': 300,
  'city': 'Quedlinburg',
  'latitude': '51.795329141617',
  'link': 'https://bundes-klinik-atlas.de/krankenhaussuche/krankenhaus/771011/',
  'longitude': '11.163533091594',
  'mail': 'info@harzklinikum.com',
  'name': 'Harzklinikum Dorothea Christiane Erxleben GmbH  Standort Quedlinburg',
  'phone': '+49 (0)3946/9090',
  'street': 'Ditfurter Weg 24',
  'zip': '06484'},
 {'beds_number': 262,
  'city': 'Wernigerode',
  'latitude': '51.835429468458',
  'link': 'https://bundes-klinik-atlas.de/krankenhaussuche/krankenhaus/771012/',
  'longitude': '10.774408936550',
  'mail': 'info@harzklinikum.com',
  'name': 'Harzklinikum Dorot

## Retrieve complete information for each hospital via a search

### Get complete list, no search criteria

In [ ]:
url = "https://bundes-klinik-atlas.de/searchresults/"

response = requests.get(
    url,
    params={
        "tx_solr_start": 0,
        "tx_solr_rows": 5,
    },
)

response.raise_for_status()

data = response.json()

data

{'results': [{'id': 773675,
   'header': 'Universitätsklinikum Hamburg-Eppendorf (UKE)',
   'address': 'Martinistraße 52, 20251 Hamburg',
   'detailLink': '/krankenhaussuche/krankenhaus/773675/?tx_tverzhospitaldata_show%5Bquantile%5D=2022%2C5764%2C10079%2C17189&cHash=1db7ecab672bad753b2e00b73e637f7e',
   'content': {'items': [{'header': 'Behandlungsfälle',
      'icon': 'icon-behandlungsfaelle',
      'tooltip': 'Hier sehen Sie, wie viele Patientinnen und Patienten innerhalb eines Jahres in diesem Krankenhaus behandelt wurden und ob das vergleichsweise viel oder wenig ist. Behandlungen psychischer Erkrankungen werden nicht abgebildet.<br><br>Weitere Informationen erhalten Sie <a href="/hilfe-informationen/#c491">hier</a>.',
      'infoData': [{'template': '\\@ce-info-table-short',
        'items': [{'template': '\\@c-tacho-text',
          'data': {'tachoData': {'type5': True, 'scale': 5},
           'text': '<strong>75.856 </strong>(sehr viele)',
           'tooltipText': ''}}]}]},
  

### Narrow down search via search criteria

In [37]:
import requests


def search_hospitals(
    icd=None,
    ops=None,
    location=None,
    start=0,
    rows=10,
):
    """Search hospitals using ICD, OPS, and geographic filters.

    Parameters
    ----------
    icd : str, optional
        ICD code used to filter hospitals by diagnosis.
    ops : str, optional
        OPS code used to filter hospitals by procedure or treatment.
    location : str, optional
        Geographic label used to filter hospitals by location.
    start : int, default=0
        Index of the first result to return.
    rows : int, default=10
        Maximum number of results to return.

    Returns
    -------
    dict
        JSON response from the Bundes-Klinik-Atlas search API.
    """
    params = {
        "tx_solr_start": start,
        "tx_solr_rows": rows,
    }

    if icd is not None:
        params["tx_solr_icd"] = icd

    if ops is not None:
        params["tx_solr_ops"] = ops

    if location is not None:
        params["tx_solr_geolabel"] = location

    response = requests.get(
        f"{configuration.host}/searchresults/",
        params=params,
    )

    response.raise_for_status()

    return response.json()